[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ray-certified/notebooks/day-04-ray-data-transforms.ipynb#scrollTo=a1b2c3d4)

---
# Day 4 · Ray Data — Datasets, Transforms, and Reading Files
**certified-journeys / ray-certified** · Ray for Distributed Python · Learn Badge

> **Goal for today:** Build fluency with Ray Data's lazy execution model — reading, transforming, filtering, and writing distributed datasets using `ray.data` APIs.


In [ ]:
%pip install -q 'ray[data]' pandas


## Step 1 · Ray Data Overview — What is it and why does it exist?

**Ray Data** is Ray's scalable data-loading and preprocessing library. It sits between your raw data and distributed training or serving code.

Key ideas:
- A **Dataset** is a distributed, lazily-evaluated sequence of records (rows or tensors).
- Ray Data automatically parallelises reads and transforms across all available CPUs.
- Unlike pandas, Ray Data does **not** materialise the full dataset in a single process — blocks are spread across the object store.
- Common sources: CSV, Parquet, JSON, images, Python lists, pandas DataFrames.

```
Data source  →  read_*()  →  Dataset (lazy)  →  .map() / .filter() / .map_batches()  →  write_*() / .to_pandas()
```

📖 Docs: https://docs.ray.io/en/latest/data/data.html


In [ ]:
import ray

# ray.init() with a small object store is fine for Colab
ray.init(ignore_reinit_error=True)

print("Ray version:", ray.__version__)
print("Cluster resources:", ray.cluster_resources())


### What just happened?

- **`ray.init()`** starts a local Ray cluster inside the Colab runtime (head node only).
- `ignore_reinit_error=True` prevents a crash if the cell is re-run — safe in notebooks.
- **`ray.cluster_resources()`** lists CPU and memory available to distribute work across.
- In production you replace `ray.init()` with `ray.init(address='auto')` to connect to an existing cluster.


## Step 2 · Creating Datasets — `from_items()` and `range()`

In Colab, fetching remote CSV/Parquet URLs can be slow or flaky. We create **synthetic data** in-process using:
- `ray.data.from_items(list_of_dicts)` — from a Python list of records
- `ray.data.range(n)` — integer range, equivalent to a column named `id`

In production you would use `ray.data.read_csv(path)` or `ray.data.read_parquet(path)` with an S3/GCS URI or local path.


In [ ]:
import ray

# ── Synthetic dataset: 1 000 employee records ──────────────────────────────
import random
random.seed(42)

departments = ["Engineering", "Sales", "Marketing", "Finance", "HR"]
records = [
    {
        "employee_id": i,
        "name": f"Employee_{i}",
        "department": random.choice(departments),
        "salary": random.randint(50_000, 150_000),
        "years": random.randint(1, 20),
    }
    for i in range(1_000)
]

# Create a Ray Dataset from the list
ds = ray.data.from_items(records)

print("Schema:", ds.schema())        # column names + dtypes
print("Row count:", ds.count())      # triggers a full pass — expensive on large data

# .show() prints the first N rows without materialising everything
ds.show(3)


### What just happened?

- **`ray.data.from_items()`** partitions the list into blocks and stores them in Ray's distributed object store.
- **`.schema()`** returns an Arrow schema with column names and dtypes — no data is moved.
- **`.count()`** is a terminal action that scans all blocks; use it deliberately on large datasets.
- **`.show(n)`** prints the first `n` rows cheaply without pulling everything to the driver.
- Production equivalent: `ray.data.read_csv("s3://bucket/employees/*.csv")` — Ray reads each file as a separate block in parallel.


## Step 3 · Reading CSV and Parquet — Parallelism Comparison

Ray Data's `read_csv()` and `read_parquet()` both produce Datasets, but **Parquet is preferred** for large-scale analytics:

| Feature | CSV | Parquet |
|---|---|---|
| Column pruning | ✗ reads all columns | ✓ skips unneeded columns |
| Row group skipping | ✗ | ✓ min/max statistics |
| Compression | optional, low ratio | Snappy/Zstd built-in |
| Parallelism unit | 1 block per file | 1 block per row group |

We demonstrate with in-memory files written to `/tmp`.


In [ ]:
import pandas as pd
import os

# Write synthetic data to disk so read_csv / read_parquet have something to load
df = pd.DataFrame(records)
os.makedirs("/tmp/ray_demo", exist_ok=True)

csv_path     = "/tmp/ray_demo/employees.csv"
parquet_path = "/tmp/ray_demo/employees.parquet"

df.to_csv(csv_path, index=False)
df.to_parquet(parquet_path, index=False)

# ── Read CSV ───────────────────────────────────────────────────────────────
ds_csv = ray.data.read_csv(csv_path)
print("CSV schema:", ds_csv.schema())
print("CSV blocks:", ds_csv.num_blocks())   # 1 block per file

# ── Read Parquet ───────────────────────────────────────────────────────────
ds_pq = ray.data.read_parquet(parquet_path)
print("Parquet schema:", ds_pq.schema())
print("Parquet blocks:", ds_pq.num_blocks()) # 1 block per row group

# Inspect first few rows of the Parquet dataset
ds_pq.show(2)


### What just happened?

- **`read_csv()`** and **`read_parquet()`** return Datasets immediately — no I/O until a terminal action.
- **`num_blocks()`** reveals the parallelism degree: more blocks → more parallel tasks.
- With Parquet, Ray can exploit **predicate pushdown** via `.filter()` before reading — saving I/O.
- In production, point these at `s3://`, `gs://`, or `abfss://` URIs; Ray handles multipart reads automatically.


## Step 4 · Row Transforms with `.map()`

**`.map(fn)`** applies `fn` to every **row** (a Python dict). Use it for lightweight, row-level transforms that don't benefit from batch vectorisation.

- Input: one dict → Output: one dict (same or different keys).
- Ray runs these tasks in parallel across blocks.
- Avoid heavy imports inside `fn`; import at the top of the function.

📖 https://docs.ray.io/en/latest/data/transforming-data.html


In [ ]:
# .map() — row-level transform
# Add a derived column: annual_bonus = 10% of salary for Engineering, 5% otherwise

def add_bonus(row: dict) -> dict:
    """Compute annual bonus based on department."""
    rate = 0.10 if row["department"] == "Engineering" else 0.05
    row["bonus"] = round(row["salary"] * rate, 2)
    return row

ds_with_bonus = ds.map(add_bonus)

# Show verifies the transform worked without materialising all 1 000 rows
ds_with_bonus.show(5)
print("Schema after .map():", ds_with_bonus.schema())


### What just happened?

- **`.map(fn)`** is still lazy — the transform is recorded but not run until `.show()` or another terminal action.
- The function signature `dict → dict` is the contract for row-level maps.
- **Ray serialises `add_bonus`** via cloudpickle and ships it to worker tasks automatically.
- For CPU-bound transforms on structured data, prefer `.map_batches()` (next step) which enables NumPy/pandas vectorisation.


## Step 5 · Batch Transforms with `.map_batches()`

**`.map_batches(fn, batch_format='pandas')`** delivers a pandas DataFrame (or NumPy dict) per block to `fn`. This is the right tool for:
- Vectorised NumPy operations
- pandas `.apply()` or column arithmetic
- Running a model on a mini-batch

| API | Input | Best for |
|---|---|---|
| `.map(fn)` | one dict | simple field transforms |
| `.map_batches(fn)` | pandas DataFrame / np array | vectorised, model inference |


In [ ]:
import pandas as pd
import numpy as np

# .map_batches() — batch-level transform with pandas
# Normalise salary to [0, 1] within each batch using min-max scaling

def normalise_salary(batch: pd.DataFrame) -> pd.DataFrame:
    """Min-max scale the salary column within each batch."""
    lo, hi = batch["salary"].min(), batch["salary"].max()
    if hi > lo:  # avoid divide-by-zero on degenerate batches
        batch["salary_norm"] = (batch["salary"] - lo) / (hi - lo)
    else:
        batch["salary_norm"] = 0.0
    return batch

ds_normalised = (
    ds
    .map_batches(
        normalise_salary,
        batch_format="pandas",   # deliver a pandas DataFrame to fn
        batch_size=256,          # rows per batch; None = one block at a time
    )
)

# Materialise to a single pandas DataFrame for inspection
sample = ds_normalised.limit(10).to_pandas()
print(sample[["employee_id", "salary", "salary_norm"]])


### What just happened?

- **`batch_format='pandas'`** tells Ray to deserialise each block into a DataFrame before calling `fn`.
- **`batch_size=256`** controls how many rows are passed per call — tune this to fit GPU memory when doing inference.
- **`.limit(10).to_pandas()`** is a common pattern for spot-checking: `.limit()` avoids processing all blocks.
- **Min-max normalisation here is per-batch**, not global. For global stats, compute them first with `.mean()` / `.std()` then pass as constants into the lambda.


## Step 6 · Chaining `.filter()` + `.map_batches()` + `.to_pandas()`

Ray Data supports **method chaining** — transforms compose into a lazy execution plan. Ray optimises the plan before running it.

Pattern:
```python
result = (
    ds
    .filter(predicate_fn)       # row-level boolean
    .map_batches(transform_fn)  # batch-level transform
    .to_pandas()                # terminal action — runs the plan
)
```

📖 https://docs.ray.io/en/latest/data/transforming-data.html


In [ ]:
# Chain: filter Engineering employees → normalise salary → materialise

def is_engineering(row: dict) -> bool:
    """Keep only Engineering department rows."""
    return row["department"] == "Engineering"

def add_seniority(batch: pd.DataFrame) -> pd.DataFrame:
    """Classify employees by years of service."""
    batch["seniority"] = pd.cut(
        batch["years"],
        bins=[0, 3, 8, float("inf")],
        labels=["Junior", "Mid", "Senior"],
    )
    return batch

eng_df = (
    ds
    .filter(is_engineering)                # drops non-Engineering rows
    .map_batches(normalise_salary, batch_format="pandas")
    .map_batches(add_seniority,    batch_format="pandas")
    .to_pandas()                           # TERMINAL: executes the full plan
)

print(f"Engineering employees: {len(eng_df)}")
print(eng_df[["name", "salary", "salary_norm", "seniority"]].head(8))
print("\nSeniority distribution:")
print(eng_df["seniority"].value_counts())


### What just happened?

- **The full chain is lazy** until `.to_pandas()` — Ray builds a DAG internally and executes it as one pipeline.
- **`.filter()`** accepts a row → bool function; rows where the function returns `False` are dropped.
- **Chaining `.map_batches()` twice** is safe — each call adds a stage to the plan.
- **`.to_pandas()`** collects all blocks to the driver node; only do this when the result fits in memory.


## Step 7 · Materialising with `.materialize()`

**`.materialize()`** forces execution and caches the result in Ray's object store. Use it when:
- You want to **reuse** the dataset across multiple downstream transforms without re-reading.
- You need to **checkpoint** mid-pipeline.
- You want to measure performance of a specific stage.

> 💡 **Tip:** Ray Data is lazy by default — transforms are not executed until you call `.to_pandas()`, `.show()`, or `.write_parquet()`. Use `.materialize()` to force execution and cache in the object store.


In [ ]:
# Materialize: execute and cache the filtered + normalised dataset
ds_cached = (
    ds
    .filter(is_engineering)
    .map_batches(normalise_salary, batch_format="pandas")
    .materialize()   # executes here; result stored in object store
)

print("Materialised dataset type:", type(ds_cached))
print("Row count (from cache):", ds_cached.count())  # cheap — reads from object store
print("Blocks in object store:", ds_cached.num_blocks())

# Subsequent operations on ds_cached reuse cached blocks
ds_cached.show(3)


### What just happened?

- **`.materialize()`** returns a new Dataset whose blocks are pinned in the object store.
- Subsequent calls (`.count()`, `.show()`, etc.) read from this cache — no re-computation.
- **Trade-off:** materialising consumes object store memory. For very large datasets, only materialise if downstream reuse justifies the memory cost.
- In ML pipelines, materialise **after expensive preprocessing** and before training loops that iterate over data multiple times.


## Step 8 · Writing Data — `write_parquet()` and `write_csv()`

Ray Data can write distributed output with:
- **`write_parquet(path)`** — each block → one Parquet file in the output directory
- **`write_csv(path)`** — each block → one CSV file in the output directory

Both functions are **terminal actions** — they trigger the full execution plan.

📖 https://docs.ray.io/en/latest/data/saving-data.html


In [ ]:
import os

out_parquet = "/tmp/ray_demo/output_parquet"
out_csv     = "/tmp/ray_demo/output_csv"

# Write Parquet — preferred for downstream analytics
ds_cached.write_parquet(out_parquet)

# Write CSV — useful for human-readable debugging output
ds_cached.write_csv(out_csv)

# Inspect written files
pq_files  = os.listdir(out_parquet)
csv_files = os.listdir(out_csv)

print(f"Parquet files written: {len(pq_files)}")
print("  ", pq_files[:3])

print(f"CSV files written: {len(csv_files)}")
print("  ", csv_files[:3])

# Round-trip: read one of the written Parquet files back
ds_rt = ray.data.read_parquet(out_parquet)
print("\nRound-trip schema:", ds_rt.schema())
ds_rt.show(2)


### What just happened?

- **`write_parquet()` and `write_csv()`** write one file per block — enabling parallel writes on distributed storage.
- The output directory is created automatically. On S3/GCS, use `s3://bucket/prefix/` as the path.
- **Round-tripping** through Parquet preserves column types exactly — CSV loses type info (all strings on read).
- In production pipelines, **always write Parquet** for intermediate steps; use CSV only for final human-readable exports.


In [ ]:
# ── Challenge ──────────────────────────────────────────────────────────────
# Challenge: Build a preprocessing pipeline for the full employee dataset:
#   1. Filter to employees with salary > 80 000
#   2. Use .map() to add a 'tenure_bonus' column:
#        tenure_bonus = years * 1000
#   3. Use .map_batches() to add a 'total_comp' column:
#        total_comp = salary + tenure_bonus
#   4. Materialise the result
#   5. Print the schema and the top 5 rows sorted by total_comp (descending)
#      Hint: materialise first, then .to_pandas().sort_values(...)

# Your solution here
def filter_high_salary(row):
    # Return True if salary > 80_000
    pass

def add_tenure_bonus(row):
    # Add tenure_bonus column
    pass

def add_total_comp(batch: pd.DataFrame) -> pd.DataFrame:
    # Add total_comp column
    pass

# Build and execute the pipeline
# result_ds = ds. ...


---
## Day 4 key concepts recap

| Concept | What to remember |
|---|---|
| `ray.data.from_items()` | Creates a Dataset from a Python list — great for testing and small synthetic data |
| `read_csv()` / `read_parquet()` | Both return lazy Datasets; Parquet enables column pruning and row group skipping |
| `.map(fn)` | Row-level transform: `dict → dict`; use for simple field derivations |
| `.map_batches(fn)` | Batch-level transform: `DataFrame → DataFrame`; use for vectorised or model operations |
| `.filter(fn)` | Row-level predicate: `dict → bool`; lazy — Ray optimises filter pushdown where possible |
| `.materialize()` | Forces execution and pins result in object store; safe to call multiple times downstream |
| `.to_pandas()` | Terminal action: collects all blocks to driver; only when result fits in memory |
| `write_parquet()` / `write_csv()` | Terminal: writes one file per block; supports S3/GCS URIs in production |

> **Tip:** Ray Data is lazy by default — transforms are not executed until you call `.to_pandas()`, `.show()`, or `.write_parquet()`. Use `.materialize()` to force execution and cache in the object store.

---
## What's next
**Day 5** → Ray Train — Distributed Model Training with PyTorch and scikit-learn. You'll wrap a PyTorch MLP in `TorchTrainer`, scale to multiple workers, and use `ScalingConfig` to control parallelism.

Mark Day 4 complete in your [tracker](../index.html).
